In [14]:
import pandas as pd

In [15]:
df = pd.read_csv("../data/raw/static/machine_measurements.csv")

/tmp/ipykernel_308/2266447113.py:1: DtypeWarning: Columns (16,17,18,19,20,21) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/static/machine_measurements.csv")


In [16]:
report_cols = [c for c in df.columns if c.startswith("report_")]
df["full_report"] = (
    df[report_cols]
    .fillna("")                # avoid "nan" strings
    .agg(" ".join, axis=1)     # concatenate per row
    .str.strip()
)


In [17]:
df = df.drop(columns = ['report_0', 'report_1',
       'report_2', 'report_3', 'report_4', 'report_5', 'report_6', 'report_7',
       'report_8', 'report_9', 'report_10', 'report_11', 'report_12',
       'report_13', 'report_14', 'report_15', 'report_16', 'report_17'])

In [18]:
#Clean text
df["report_clean"] = (
    df["full_report"]
    .str.lower()
    .str.replace(r"[^a-z\s]", "", regex=True)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)


In [19]:
RHYTHM_LABELS = {
    "sinus_rhythm": [
        "sinus rhythm",
        "sinus rhythm normal"
    ],
    "sinus_bradycardia": [
        "sinus bradycardia",
        "bradycardia"
    ],
    "sinus_tachycardia": [
        "sinus tachycardia",
        "tachycardia"
    ],
    "atrial_fibrillation": [
        "atrial fibrillation",
        "fibrillation"
    ]
}


CONDUCTION_LABELS = {
    "bundle_branch_block": [
        "bundle branch block",
        "branch block"
    ],
    "av_block": [
        "av block",
        "degree av block",
        "st degree av block"
    ],
    "left_anterior_fascicular_block": [
        "left anterior fascicular block",
        "anterior fascicular block"
    ]
}



STRUCTURAL_LABELS = {
    "left_ventricular_hypertrophy": [
        "left ventricular hypertrophy",
        "ventricular hypertrophy"
    ],
    "axis_deviation": [
        "axis deviation",
        "left axis deviation"
    ],
    "low_qrs_voltage": [
        "low qrs voltages",
        "low qrs",
        "low qrs voltages precordial"
    ]
}


REPOLARIZATION_LABELS = {
    "st_t_abnormality": [
        "stt changes",
        "stt changes nonspecific",
        "wave changes",
        "wave changes nonspecific",
        "t wave changes"
    ],
    "poor_r_wave_progression": [
        "poor wave progression"
    ],
    "st_elevation_depression": [
        "st changes",
        "st degree"
    ]
}


ISCHEMIA_LABELS = {
    "myocardial_ischemia": [
        "ischemia",
        "myocardial ischemia"
    ],
    "infarct_pattern": [
        "inferior infarct",
        "anterior infarct",
        "infarct age undetermined"
    ]
}


ECTOPY_LABELS = {
    "pvcs": [
        "pvcs"
    ]
}


IMPRESSION_LABELS = {
    "normal_ecg": [
        "normal ecg",
        "probable normal",
        "normal variant",
        "probable normal variant"
    ],
    "borderline_ecg": [
        "borderline ecg",
        "nonspecific borderline ecg"
    ],
    "abnormal_ecg": [
        "abnormal ecg",
        "nonspecific abnormal ecg",
        "undetermined abnormal ecg"
    ]
}


CANONICAL_MAP = {
    **RHYTHM_LABELS,
    **CONDUCTION_LABELS,
    **STRUCTURAL_LABELS,
    **REPOLARIZATION_LABELS,
    **ISCHEMIA_LABELS,
    **ECTOPY_LABELS,
    **IMPRESSION_LABELS
}

#ADDITIONS
PACED_RHYTHM_LABELS = {
    "atrial_paced_rhythm": [
        "atrial paced rhythm",
        "atrial paced"
    ],
    "ventricular_paced_rhythm": [
        "ventricular paced rhythm",
        "v paced",
        "ventricular paced"
    ],
    "dual_paced_rhythm": [
        "dual paced rhythm",
        "a v dual paced",
        "atrial ventricular dual paced"
    ]
}


ADDITIONAL_RHYTHM_LABELS = {
    "sinus_arrhythmia": [
        "sinus arrhythmia",
        "slow sinus arrhythmia"
    ],
    "ectopic_atrial_rhythm": [
        "ectopic atrial rhythm",
        "wandering atrial pacemaker"
    ],
    "junctional_rhythm": [
        "junctional rhythm"
    ]
}

ADDITIONAL_FINDINGS = {
    "prolonged_qt": [
        "prolonged qt",
        "prolonged qt interval"
    ],
    "intraventricular_conduction_delay": [
        "intraventricular conduction delay"
    ],
    "right_bundle_branch_block": [
        "rbbb",
        "right bundle branch block"
    ],
    
    "abnormal_r_wave_progression": [
    r"poor\s+r[-\s]?wave\s+progression",
    r"abnormal\s+r[-\s]?wave\s+progression",
    r"delayed\s+r[-\s]?wave\s+progression",
    r"poor\s+wave\s+progression",
]

}

CANONICAL_MAP = {
    **CANONICAL_MAP,
    **PACED_RHYTHM_LABELS,
    **ADDITIONAL_RHYTHM_LABELS,
    **ADDITIONAL_FINDINGS
}


In [20]:
def contains_any(text, phrases):
    return any(p in text for p in phrases)

for label, phrases in CANONICAL_MAP.items():
    df[f"label_{label}"] = (
        df["report_clean"]
        .apply(lambda x: contains_any(x, phrases))
        .astype(int)
    )


In [21]:
#Drop dead columns
dead_cols = [
    'label_poor_r_wave_progression',
    'label_atrial_paced_rhythm',
    'label_ventricular_paced_rhythm',
    'label_dual_paced_rhythm',
    'label_abnormal_r_wave_progression'
]

df = df.drop(columns=dead_cols, errors='ignore')
#Re-add them using full-report regex
df['label_atrial_paced_rhythm'] = df['full_report'].str.contains(
    r"\batrial[-\s]?paced\b",
    case=False,
    na=False
).astype(int)

df['label_ventricular_paced_rhythm'] = df['full_report'].str.contains(
    r"\bventricular[-\s]?paced\b",
    case=False,
    na=False
).astype(int)

df['label_dual_paced_rhythm'] = df['full_report'].str.contains(
    r"\b(dual|a[-\s]?v)[-\s]?paced\b",
    case=False,
    na=False
).astype(int)

df['label_abnormal_r_wave_progression'] = df['full_report'].str.contains(
    r"\b(poor|abnormal)\s+r[-\s]?wave\s+progression\b",
    case=False,
    na=False
).astype(int)


/tmp/ipykernel_308/3570038439.py:24: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['label_dual_paced_rhythm'] = df['full_report'].str.contains(
/tmp/ipykernel_308/3570038439.py:30: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  df['label_abnormal_r_wave_progression'] = df['full_report'].str.contains(


In [22]:
#Strip label name
df = df.rename(columns=lambda c: c.replace("label_", "", 1) if c.startswith("label_") else c)

In [23]:
df

,subject_id,study_id,cart_id,ecg_time,bandwidth,filtering,rr_interval,p_onset,p_end,qrs_onset,...,sinus_arrhythmia,ectopic_atrial_rhythm,junctional_rhythm,prolonged_qt,intraventricular_conduction_delay,right_bundle_branch_block,atrial_paced_rhythm,ventricular_paced_rhythm,dual_paced_rhythm,abnormal_r_wave_progression
0,10000032,40689238,6848296,2180-07-23 08:44:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,128,170,...,0,0,0,0,0,0,0,0,0,0
1,10000032,44458630,6848296,2180-07-23 09:54:00,0.005-150 Hz,60 Hz notch Baseline filter,722,40,124,162,...,0,0,0,0,0,0,0,0,0,0
2,10000032,49036311,6376932,2180-08-06 09:07:00,0.005-150 Hz,60 Hz notch Baseline filter,600,40,130,162,...,0,0,0,0,0,0,0,0,0,0
3,10000117,45090959,6214760,2181-03-04 17:14:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,146,180,...,0,0,0,0,0,0,0,0,0,0
4,10000117,48446569,6632385,2183-09-18 13:52:00,0.0005-150 Hz,<not specified>,659,368,29999,504,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
800030,19999840,48683947,6283529,2164-09-12 12:28:00,0.005-150 Hz,60 Hz notch Baseline filter,659,40,154,182,...,0,0,0,0,0,0,0,0,0,0
800031,19999840,41842293,6947865,2164-09-17 11:31:00,0.005-150 Hz,60 Hz notch Baseline filter,967,29999,29999,200,...,0,0,0,0,0,0,0,0,0,0
800032,19999987,41190887,6848296,2145-11-02 19:54:00,0.005-150 Hz,60 Hz notch Baseline filter,526,40,148,204,...,0,0,0,0,0,0,0,0,0,0
800033,19999987,45828463,6315641,2145-11-03 03:00:00,0.005-150 Hz,60 Hz notch Baseline filter,631,40,156,202,...,0,0,0,0,0,0,0,0,0,0
